In [1]:
# Test 6 : inférence sur GPU pour bien plus de rapidité
# La GPU est 10-100x plus rapide que le CPU pour les LLMs

import torch
from transformers import pipeline
import time

# Vérifier si une GPU est disponible
print("="*60)
print("VÉRIFICATION DE LA GPU")
print("="*60)
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Nombre de GPUs : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU utilisée : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  Aucune GPU détectée. L'inférence se fera sur CPU.")
print()

# Prompt simple
prompt = "Explique-moi ce qu'est un Large Language Model (LLM) dans le domaine de l'intelligence artificielle. Un LLM est un modèle de langue basé sur le machine learning. Réponds en 2-3 phrases simples."

# Charger le modèle avec device_map="auto" pour utiliser la GPU si disponible
print("="*60)
print("CHARGEMENT DU MODÈLE")
print("="*60)
start_load = time.time()

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    tokenizer="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"  # ← Force l'utilisation de la GPU si disponible
)

load_time = time.time() - start_load
print(f"Temps de chargement : {load_time:.2f}s")
print()

# Génération avec mesure de temps
print("="*60)
print("GÉNÉRATION DE TEXTE")
print("="*60)
start_gen = time.time()

response = generator(
    prompt,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    num_return_sequences=1
)

gen_time = time.time() - start_gen
print(f"Temps de génération : {gen_time:.2f}s")
print()

# Affichage du résultat
print("="*60)
print("RÉPONSE")
print("="*60)
print(response[0]["generated_text"])
print()

# Résumé des perfs
print("="*60)
print("RÉSUMÉ DES PERFORMANCES")
print("="*60)
if torch.cuda.is_available():
    print(f"✅ Inférence sur GPU : {gen_time:.2f}s")
    print("(Avec CPU, ce serait ~10-100x plus lent)")
else:
    print(f"⚠️  Inférence sur CPU : {gen_time:.2f}s")
    print("Si tu as une GPU NVIDIA, installe CUDA pour accélérer")


VÉRIFICATION DE LA GPU
GPU disponible : True
Nombre de GPUs : 1
GPU utilisée : Quadro P1000
Mémoire GPU : 4.29 GB

CHARGEMENT DU MODÈLE


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Temps de chargement : 3.00s

GÉNÉRATION DE TEXTE


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Temps de génération : 10.82s

RÉPONSE
Explique-moi ce qu'est un Large Language Model (LLM) dans le domaine de l'intelligence artificielle. Un LLM est un modèle de langue basé sur le machine learning. Réponds en 2-3 phrases simples. Il peut comprendre et produire des langues différentes, c'est-à-dire être capable d'interpréter et traduire une variété de langues.

Un large modèle est un système de langage qui est construit avec la technologie de l'intelligence artificielle pour produire de grandes quantités de texte et interpréter les langues. Il peut comprendre et produire une variété de langues, y compris les langues étrangères ou linguistique à base de données. Il peut aussi comprendre et interpréter une grande quantité de contenu, comme les textes, les images, les vidéos et même des documents numériques. Un large modèle est souvent utilisé pour

RÉSUMÉ DES PERFORMANCES
✅ Inférence sur GPU : 10.82s
(Avec CPU, ce serait ~10-100x plus lent)
